In [0]:
from pyspark.sql.functions import *

silver_calls = spark.table("workspace.default.silver_calls")

In [0]:
daily_calls = (
    silver_calls
        .groupBy(to_date("event_time").alias("call_date"))
        .agg(
            count("*").alias("total_calls")
        )
)

In [0]:
display(daily_calls)

call_date,total_calls
2026-07-11,144


In [0]:
(
    daily_calls.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("workspace.default.gold_daily_calls")
)

In [0]:
sentiment_kpi = (
    silver_calls
        .groupBy("sentiment")
        .agg(
            count("*").alias("total_calls")
        )
)

display(sentiment_kpi)

(
    sentiment_kpi.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("workspace.default.gold_sentiment_distribution")
)

sentiment,total_calls
Neutral,50
Positive,40
Negative,54


In [0]:
agent_kpi = (
    silver_calls
        .groupBy("agent")
        .agg(
            count("*").alias("total_calls"),
            avg("duration_seconds").alias("avg_duration")
        )
)

display(agent_kpi)

(
    agent_kpi.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("workspace.default.gold_agent_performance")
)

agent,total_calls,avg_duration
Agent_19,7,793.4285714285714
Agent_16,14,631.7857142857143
Agent_7,12,486.5833333333333
Agent_17,9,579.3333333333334
Agent_2,6,464.0
Agent_13,9,565.5555555555555
Agent_10,5,824.6
Agent_18,10,688.5
Agent_3,5,445.8
Agent_11,11,728.9090909090909


In [0]:
customers = spark.table("workspace.default.silver_customers")

subscription_kpi = (
    silver_calls.alias("c")
        .join(
            customers.alias("u"),
            col("c.customer_id") == col("u.customer_id")
        )
        .groupBy(
            "subscription_type",
            "sentiment"
        )
        .agg(
            count("*").alias("total_calls")
        )
)

display(subscription_kpi)

(
    subscription_kpi.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("workspace.default.gold_subscription_sentiment")
)

subscription_type,sentiment,total_calls
Basic,Neutral,2
Premium,Neutral,6
Basic,Negative,7
Basic,Positive,7
Premium,Positive,5
Premium,Negative,5


In [0]:
high_risk = (
    silver_calls
        .filter(col("sentiment") == "Negative")
        .groupBy("customer_id")
        .agg(
            count("*").alias("negative_calls")
        )
        .filter(col("negative_calls") >= 2)
)

display(high_risk)

(
    high_risk.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("workspace.default.gold_high_risk_customers")
)

customer_id,negative_calls
58,2
67,2
71,2
8,2
72,2
89,2
76,2
45,2
69,2
21,2
